# Predicting 30-Day Patient Readmission
BrightLearn ML case study. Sections match the five tasks. Run top to bottom.

In [1]:
import pandas as pd, numpy as np, json, warnings, os
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, cross_val_predict, GridSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
sns.set_theme(style='whitegrid'); os.makedirs('figs', exist_ok=True); R = {}

## Task 1: Data preprocessing

In [2]:
df = pd.read_csv("Expanded_Patient_Readmission_Data_1.csv")
print(df.shape); print(df.dtypes); display(df.head()); display(df.describe(include='all').T)
num = ["Age","Length of Stay","Number of Diagnoses","Blood Pressure","Blood Sugar Levels","Previous Admissions"]
cat = ["Gender","Admission Type"]

(3000, 10)
Patient ID             int64
Age                    int64
Gender                   str
Admission Type           str
Length of Stay         int64
Number of Diagnoses    int64
Blood Pressure         int64
Blood Sugar Levels     int64
Previous Admissions    int64
Readmission              str
dtype: object


,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
0,1,62,Female,Elective,4,5,110,130,1,No
1,2,65,Male,Emergency,19,2,157,81,4,No
2,3,82,Female,Emergency,18,4,74,84,0,No
3,4,85,Male,Emergency,2,4,106,85,4,No
4,5,85,Female,Elective,19,3,80,119,3,No


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Patient ID,3000.0,NaN,NaN,NaN,1500.5,866.169729,1.0,750.75,1500.5,2250.25,3000.0
Age,3000.0,NaN,NaN,NaN,53.453,20.892996,18.0,35.0,53.0,72.0,89.0
Gender,3000,2,Male,1555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Admission Type,3000,2,Elective,1563,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Length of Stay,3000.0,NaN,NaN,NaN,15.226333,8.264858,1.0,8.0,15.0,22.0,29.0
Number of Diagnoses,3000.0,NaN,NaN,NaN,4.956667,2.575073,1.0,3.0,5.0,7.0,9.0
Blood Pressure,3000.0,NaN,NaN,NaN,119.058,34.089265,60.0,89.75,119.0,147.25,179.0
Blood Sugar Levels,3000.0,NaN,NaN,NaN,133.536,37.626311,51.0,101.0,133.0,167.0,199.0
Previous Admissions,3000.0,NaN,NaN,NaN,1.958333,1.401637,0.0,1.0,2.0,3.0,4.0
Readmission,3000,2,No,2134,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Audit before any change
R['shape']=df.shape; R['missing']=int(df.isna().sum().sum()); R['dupe_rows']=int(df.duplicated().sum()); R['dupe_ids']=int(df['Patient ID'].duplicated().sum())
R['ranges']={c:[int(df[c].min()),int(df[c].max())] for c in num+['Patient ID']}
R['labels']={c:sorted(df[c].unique().tolist()) for c in cat+['Readmission']}
R['ws']={c:int((df[c]!=df[c].str.strip()).sum()) for c in cat+['Readmission']}
R['id_seq']=bool((df['Patient ID'].sort_values().values==np.arange(1,3001)).all())
print('Missing per column:'); print(df.isna().sum())
print('Duplicate rows:',R['dupe_rows'],'| Duplicate Patient IDs:',R['dupe_ids'],'| IDs run 1..3000:',R['id_seq'])
print('Unique labels:',R['labels']); print('Labels with stray spaces:',R['ws']); print('Ranges:',R['ranges'])

Missing per column:
Patient ID             0
Age                    0
Gender                 0
Admission Type         0
Length of Stay         0
Number of Diagnoses    0
Blood Pressure         0
Blood Sugar Levels     0
Previous Admissions    0
Readmission            0
dtype: int64
Duplicate rows: 0 | Duplicate Patient IDs: 0 | IDs run 1..3000: True
Unique labels: {'Gender': ['Female', 'Male'], 'Admission Type': ['Elective', 'Emergency'], 'Readmission': ['No', 'Yes']}
Labels with stray spaces: {'Gender': 0, 'Admission Type': 0, 'Readmission': 0}
Ranges: {'Age': [18, 89], 'Length of Stay': [1, 29], 'Number of Diagnoses': [1, 9], 'Blood Pressure': [60, 179], 'Blood Sugar Levels': [51, 199], 'Previous Admissions': [0, 4], 'Patient ID': [1, 3000]}


In [4]:
# Outliers: IQR rule
out={}
for c in num:
    q1,q3=df[c].quantile([.25,.75]); i=q3-q1
    out[c]=int(((df[c]<q1-1.5*i)|(df[c]>q3+1.5*i)).sum())
R['outliers']=out; R['skew']={c:round(float(df[c].skew()),3) for c in num}
print('IQR outlier counts:',out); print('Skew:',R['skew'])
units={'Age':'years','Length of Stay':'days','Number of Diagnoses':'count','Blood Pressure':'mmHg','Blood Sugar Levels':'mg/dL','Previous Admissions':'count'}
fig,axs=plt.subplots(2,3,figsize=(11,6))
for a,c in zip(axs.ravel(),num):
    sns.boxplot(y=df[c],ax=a,color='#8FB3D9'); a.set_title(c); a.set_ylabel(f"{c} ({units[c]})")
fig.suptitle('Box plots used for the outlier check',y=1.0); plt.tight_layout(); plt.savefig('figs/f1.png',dpi=150); plt.show()

IQR outlier counts: {'Age': 0, 'Length of Stay': 0, 'Number of Diagnoses': 0, 'Blood Pressure': 0, 'Blood Sugar Levels': 0, 'Previous Admissions': 0}
Skew: {'Age': 0.01, 'Length of Stay': -0.057, 'Number of Diagnoses': 0.015, 'Blood Pressure': 0.021, 'Blood Sugar Levels': 0.021, 'Previous Admissions': 0.047}


In [5]:
# Target, encoding, split (split BEFORE fitting anything)
df['y']=(df['Readmission']=='Yes').astype(int)
R['target']={int(k):int(v) for k,v in df['y'].value_counts().items()}; R['prev']=float(df.y.mean())
X=df[num+cat]; y=df['y']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
R['split']={'train':[len(Xtr),float(ytr.mean())*100,int(ytr.sum())],'test':[len(Xte),float(yte.mean())*100,int(yte.sum())]}
print(R['split'])
feat_num, feat_cat = num, cat   # all candidate features for now; narrowed after Task 3
def make_pre():
    return ColumnTransformer([
        ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),feat_num),
        ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(drop='first',handle_unknown='ignore'))]),feat_cat)])
def pipe(m): return Pipeline([('prep',make_pre()),('clf',m)])

{'train': [2400, 28.875, 693], 'test': [600, 28.833333333333332, 173]}


## Task 2: Exploratory data analysis

In [6]:
R['desc']=df[num].agg(['mean','std','min','max']).T.round(2).to_dict('index'); display(df[num].describe().T.round(2))
fig,ax=plt.subplots(figsize=(5,4)); vc=df['Readmission'].value_counts()
b=ax.bar(vc.index,vc.values,color=['#4C72B0','#DD8452'])
for r,v in zip(b,vc.values): ax.text(r.get_x()+r.get_width()/2,v+20,f"{v} ({v/3000*100:.1f}%)",ha='center')
ax.set_title('Readmission within 30 days'); ax.set_xlabel('Readmitted within 30 days'); ax.set_ylabel('Number of admissions'); ax.set_ylim(0,2500); plt.tight_layout(); plt.savefig('figs/f2.png',dpi=150); plt.show()
fig,axs=plt.subplots(2,3,figsize=(11,6))
for a,c in zip(axs.ravel(),num):
    sns.histplot(df[c],bins=min(30,df[c].nunique()),ax=a,color='#4C72B0',discrete=df[c].nunique()<12); a.set_title(c); a.set_xlabel(f"{c} ({units[c]})"); a.set_ylabel('Admissions')
fig.suptitle('Distributions of numeric variables',y=1.0); plt.tight_layout(); plt.savefig('figs/f3.png',dpi=150); plt.show()
fig,axs=plt.subplots(1,3,figsize=(11,3.8))
for a,c in zip(axs,cat+['Readmission']):
    v=df[c].value_counts(); a.bar(v.index,v.values,color='#55A868'); a.set_title(c); a.set_xlabel(c); a.set_ylabel('Number of admissions'); a.set_ylim(0,max(v.values)*1.15)
    for i,(k,val) in enumerate(v.items()): a.text(i,val+15,str(val),ha='center')
fig.suptitle('Categorical variable frequencies',y=1.0); plt.tight_layout(); plt.savefig('figs/f4.png',dpi=150); plt.show()
fig,axs=plt.subplots(1,2,figsize=(9,4)); rates={}
for a,c in zip(axs,cat):
    g=df.groupby(c)['y'].mean()*100; rates[c]=g.round(2).to_dict()
    a.bar(g.index,g.values,color='#DD8452'); a.set_title(f'Readmission rate by {c}'); a.set_xlabel(c); a.set_ylabel('Readmission rate (%)'); a.set_ylim(0,45)
    for i,v in enumerate(g.values): a.text(i,v+.7,f"{v:.1f}%",ha='center')
plt.tight_layout(); plt.savefig('figs/f5.png',dpi=150); plt.show(); R['rates']=rates; print(rates)
fig,axs=plt.subplots(1,4,figsize=(13,4))
for a,c in zip(axs,["Age","Length of Stay","Blood Pressure","Blood Sugar Levels"]):
    sns.boxplot(data=df,x='Readmission',y=c,ax=a,palette=['#4C72B0','#DD8452']); a.set_title(c); a.set_xlabel('Readmitted within 30 days'); a.set_ylabel(f"{c} ({units[c]})")
fig.suptitle('Numeric variables by readmission outcome',y=1.0); plt.tight_layout(); plt.savefig('figs/f6.png',dpi=150); plt.show()
fig,axs=plt.subplots(1,2,figsize=(10,4)); r7={}
for a,c in zip(axs,["Number of Diagnoses","Previous Admissions"]):
    g=df.groupby(c)['y'].mean()*100; r7[c]=g.round(2).to_dict()
    a.bar(g.index.astype(str),g.values,color='#8172B3'); a.set_title(f'Readmission rate by {c}'); a.set_xlabel(c+' (count)'); a.set_ylabel('Readmission rate (%)'); a.set_ylim(0,45); a.axhline(df.y.mean()*100,ls='--',c='grey')
plt.tight_layout(); plt.savefig('figs/f7.png',dpi=150); plt.show(); R['r7']=r7; print(r7)
cm=df[num].corr().round(3); R['corr_target']=df[num+['y']].corr()['y'].round(3).to_dict(); R['max_pair_corr']=float(cm.where(~np.eye(len(cm),dtype=bool)).abs().max().max())
fig,ax=plt.subplots(figsize=(7.5,5.8)); sns.heatmap(df[num+['y']].rename(columns={'y':'Readmitted (1=Yes)'}).corr(),annot=True,fmt='.2f',cmap='coolwarm',center=0,vmin=-1,vmax=1,ax=ax); ax.set_title('Correlation matrix (Pearson r)'); plt.tight_layout(); plt.savefig('figs/f8.png',dpi=150); plt.show()
print('Largest correlation between two predictors:',R['max_pair_corr']); print('Correlation with readmission:',R['corr_target'])

,count,mean,std,min,25%,50%,75%,max
Age,3000.0,53.45,20.89,18.0,35.00,53.0,72.00,89.0
Length of Stay,3000.0,15.23,8.26,1.0,8.00,15.0,22.00,29.0
Number of Diagnoses,3000.0,4.96,2.58,1.0,3.00,5.0,7.00,9.0
Blood Pressure,3000.0,119.06,34.09,60.0,89.75,119.0,147.25,179.0
Blood Sugar Levels,3000.0,133.54,37.63,51.0,101.00,133.0,167.00,199.0
Previous Admissions,3000.0,1.96,1.40,0.0,1.00,2.0,3.00,4.0


{'Gender': {'Female': 28.17, 'Male': 29.52}, 'Admission Type': {'Elective': 29.37, 'Emergency': 28.32}}


{'Number of Diagnoses': {1: 30.84, 2: 27.22, 3: 31.72, 4: 26.42, 5: 25.0, 6: 29.68, 7: 25.07, 8: 31.64, 9: 32.7}, 'Previous Admissions': {0: 30.94, 1: 27.13, 2: 27.93, 3: 30.41, 4: 27.99}}
Largest correlation between two predictors: 0.028
Correlation with readmission: {'Age': 0.029, 'Length of Stay': 0.021, 'Number of Diagnoses': 0.009, 'Blood Pressure': 0.01, 'Blood Sugar Levels': 0.017, 'Previous Admissions': -0.007, 'y': 1.0}


## Task 3: Feature selection (training data only)

In [7]:
tr=Xtr.copy(); tr['y']=ytr.values; fs={}
for c in cat:
    ct=pd.crosstab(tr[c],tr['y']); chi,p,_,_=stats.chi2_contingency(ct); v=np.sqrt(chi/ct.values.sum())
    fs[c]={'stat':f"chi2={chi:.2f}",'p':p,'eff':f"V={v:.3f}"}
for c in num:
    a=tr.loc[tr.y==1,c]; b_=tr.loc[tr.y==0,c]; t,p=stats.ttest_ind(a,b_,equal_var=False); d=(a.mean()-b_.mean())/np.sqrt((a.var()+b_.var())/2)
    fs[c]={'stat':f"t={t:.2f}",'p':p,'eff':f"d={d:.3f}"}
enc=pd.get_dummies(Xtr,columns=cat,drop_first=True)
disc=[c in ['Number of Diagnoses','Previous Admissions','Gender_Male','Admission Type_Emergency','Age','Length of Stay'] for c in enc.columns]
miD=dict(zip(enc.columns,mutual_info_classif(enc,ytr,discrete_features=disc,random_state=42)))
fs['Gender']['mi']=miD['Gender_Male']; fs['Admission Type']['mi']=miD['Admission Type_Emergency']
for c in num: fs[c]['mi']=miD[c]
# permutation importance on a held-back part of the TRAINING data
Xa,Xv,ya,yv=train_test_split(Xtr,ytr,test_size=.25,stratify=ytr,random_state=1)
rf=pipe(RandomForestClassifier(n_estimators=300,min_samples_leaf=20,max_depth=6,random_state=42,class_weight='balanced')).fit(Xa,ya)
pi=permutation_importance(rf,Xv,yv,scoring='roc_auc',n_repeats=30,random_state=42)
for c,m_,s_ in zip(Xa.columns,pi.importances_mean,pi.importances_std): fs[c]['pi']=float(m_); fs[c]['pis']=float(s_)
R['rf_inner_auc']=[float(roc_auc_score(ya,rf.predict_proba(Xa)[:,1])),float(roc_auc_score(yv,rf.predict_proba(Xv)[:,1]))]
# for comparison: importance measured on the rows the forest was fitted on
pit=permutation_importance(rf,Xa,ya,scoring='roc_auc',n_repeats=10,random_state=42)
R['pi_fit_range']=[float(pit.importances_mean.min()),float(pit.importances_mean.max())]
l1=pipe(LogisticRegression(penalty='l1',solver='liblinear',C=0.1,class_weight='balanced')).fit(Xtr,ytr)
names=num+['Gender_Male','Admission Type_Emergency']; cd=dict(zip(names,l1.named_steps['clf'].coef_[0]))
for c in num: fs[c]['l1']=float(cd[c])
fs['Gender']['l1']=float(cd['Gender_Male']); fs['Admission Type']['l1']=float(cd['Admission Type_Emergency'])
R['fs']={k:{kk:(float(vv) if isinstance(vv,(np.floating,float)) else vv) for kk,vv in v.items()} for k,v in fs.items()}
display(pd.DataFrame(R['fs']).T)
print('Random forest AUC on fitted rows / held-back rows:',R['rf_inner_auc']); print('Importance range when measured on fitted rows:',R['pi_fit_range'])
o=sorted(Xa.columns,key=lambda c:fs[c]['pi'])
fig,ax=plt.subplots(figsize=(8,4)); ax.barh(o,[fs[c]['pi'] for c in o],xerr=[fs[c]['pis'] for c in o],color='#4C72B0'); ax.axvline(0,c='k',lw=.8)
ax.set_xlabel('Drop in ROC-AUC when the feature is shuffled'); ax.set_ylabel('Feature'); ax.set_title('Permutation importance (random forest, held-back rows)'); plt.tight_layout(); plt.savefig('figs/f9.png',dpi=150); plt.show()

,stat,p,eff,mi,pi,pis,l1
Gender,chi2=1.02,0.31356,V=0.021,0.000231,0.002789,0.006867,0.04447
Admission Type,chi2=2.49,0.114304,V=0.032,0.00055,-0.006515,0.012036,-0.067025
Age,t=1.23,0.220416,d=0.055,0.014523,0.009246,0.010029,0.038386
Length of Stay,t=0.81,0.418067,d=0.036,0.00784,0.020114,0.016321,0.019885
Number of Diagnoses,t=0.57,0.570604,d=0.026,0.001259,0.006832,0.007094,0.007523
Blood Pressure,t=0.11,0.912666,d=0.005,0.0,-0.004247,0.00824,0.0
Blood Sugar Levels,t=0.72,0.468757,d=0.032,0.0,0.02811,0.014254,0.017488
Previous Admissions,t=-0.21,0.832838,d=-0.010,0.00061,0.011745,0.01083,0.0


Random forest AUC on fitted rows / held-back rows: [0.8037019230769231, 0.529409375803766]
Importance range when measured on fitted rows: [0.028763671875000008, 0.10901171875000007]


## Task 4: Model building (cross-validation on training data only)

In [8]:
# Final feature list from the keep/drop decisions in Task 3: Patient ID, Gender and Blood Pressure are dropped
feat_num=["Age","Length of Stay","Number of Diagnoses","Blood Sugar Levels","Previous Admissions"]; feat_cat=["Admission Type"]
print('Features used for modelling:',feat_num+feat_cat)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
mk={'Baseline (always predict No)':DummyClassifier(strategy='most_frequent'),
    'Logistic regression':LogisticRegression(max_iter=1000,class_weight='balanced'),
    'Decision tree':DecisionTreeClassifier(class_weight='balanced',random_state=42),
    'Random forest':RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42,n_jobs=-1)}
cvres={}
for n,m in mk.items():
    r=cross_validate(pipe(m),Xtr,ytr,cv=cv,scoring='roc_auc',return_train_score=True); cvres[n]=[float(r['test_score'].mean()),float(r['test_score'].std()),float(r['train_score'].mean())]
    print(f"{n:32s} CV AUC {cvres[n][0]:.3f} +/- {cvres[n][1]:.3f} | train AUC {cvres[n][2]:.3f}")
R['cv']=cvres
sm=ImbPipeline([('prep',make_pre()),('sm',SMOTE(random_state=42)),('clf',LogisticRegression(max_iter=1000))])
s=cross_val_score(sm,Xtr,ytr,cv=cv,scoring='roc_auc'); R['smote_lr']=[float(s.mean()),float(s.std())]; print('SMOTE inside CV (logistic regression):',R['smote_lr'])

Features used for modelling: ['Age', 'Length of Stay', 'Number of Diagnoses', 'Blood Sugar Levels', 'Previous Admissions', 'Admission Type']
Baseline (always predict No)     CV AUC 0.500 +/- 0.000 | train AUC 0.500


Logistic regression              CV AUC 0.501 +/- 0.033 | train AUC 0.535


Decision tree                    CV AUC 0.500 +/- 0.022 | train AUC 1.000


Random forest                    CV AUC 0.525 +/- 0.014 | train AUC 1.000
SMOTE inside CV (logistic regression): [0.5059897317030092, 0.02759074954295805]


In [9]:
grids={'Logistic regression':(LogisticRegression(max_iter=1000,class_weight='balanced'),{'clf__C':[0.001,0.01,0.1,1,10,100]}),
 'Decision tree':(DecisionTreeClassifier(class_weight='balanced',random_state=42),{'clf__max_depth':[2,3,4,5,8],'clf__min_samples_leaf':[10,25,50,100]}),
 'Random forest':(RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42,n_jobs=-1),{'clf__max_depth':[3,5,8],'clf__min_samples_leaf':[10,25,50]})}
tuned={}; R['tune']={}
for n,(m,g) in grids.items():
    gs=GridSearchCV(pipe(m),g,cv=cv,scoring='roc_auc',n_jobs=-1,return_train_score=True).fit(Xtr,ytr); i=gs.best_index_; tuned[n]=gs.best_estimator_
    R['tune'][n]={'grid':{k.replace('clf__',''):v for k,v in g.items()},'best':{k.replace('clf__',''):v for k,v in gs.best_params_.items()},'untuned':cvres[n][0],'tuned':float(gs.best_score_),'tuned_sd':float(gs.cv_results_['std_test_score'][i]),'train':float(gs.cv_results_['mean_train_score'][i]),'n':int(np.prod([len(v) for v in g.values()]))}
    print(n,R['tune'][n]['best'],f"CV AUC {R['tune'][n]['tuned']:.3f} +/- {R['tune'][n]['tuned_sd']:.3f} (untuned {cvres[n][0]:.3f})")
best=max(tuned,key=lambda n:R['tune'][n]['tuned']); R['final']=best; print('Highest tuned CV AUC:',best)

Logistic regression {'C': 100} CV AUC 0.501 +/- 0.033 (untuned 0.501)


Decision tree {'max_depth': 5, 'min_samples_leaf': 100} CV AUC 0.525 +/- 0.015 (untuned 0.500)


Random forest {'max_depth': 8, 'min_samples_leaf': 10} CV AUC 0.522 +/- 0.015 (untuned 0.525)
Highest tuned CV AUC: Decision tree


In [10]:
# Choose threshold from out-of-fold predictions on TRAINING data (test set not used)
oof=cross_val_predict(clone(tuned[best]),Xtr,ytr,cv=cv,method='predict_proba')[:,1]
ths=np.linspace(.05,.95,91); rec=[recall_score(ytr,oof>=t) for t in ths]
cand=[t for t,r in zip(ths,rec) if r>=.70]; thr=float(max(cand)); R['thr']=thr; R['oof_auc']=float(roc_auc_score(ytr,oof))
R['thr_oof']={'recall':float(recall_score(ytr,oof>=thr)),'precision':float(precision_score(ytr,oof>=thr))}; print('Threshold',thr,R['thr_oof'],'OOF AUC',R['oof_auc'])

Threshold 0.4499999999999999 {'recall': 0.7027417027417028, 'precision': 0.2962287104622871} OOF AUC 0.5235305604374145


## Task 5: Evaluation (test set used once)

In [11]:
final=tuned[best].fit(Xtr,ytr)
def met(m,t=.5):
    p=m.predict_proba(Xte)[:,1]; yp=(p>=t).astype(int)
    return dict(acc=accuracy_score(yte,yp),prec=precision_score(yte,yp,zero_division=0),rec=recall_score(yte,yp),f1=f1_score(yte,yp),auc=roc_auc_score(yte,p),tn_fp_fn_tp=confusion_matrix(yte,yp).ravel().tolist())
T={'Baseline (always predict No)':met(pipe(mk['Baseline (always predict No)']).fit(Xtr,ytr))}
for n in tuned: T[n+' (tuned)']=met(tuned[n])
R['test']=T; R['test_final_thr']=met(final,thr); display(pd.DataFrame(T).T.drop(columns='tn_fp_fn_tp').round(3)); print('Final model at chosen threshold:',R['test_final_thr'])
R['gap']={n:[float(roc_auc_score(ytr,m.predict_proba(Xtr)[:,1])),float(roc_auc_score(yte,m.predict_proba(Xte)[:,1]))] for n,m in tuned.items()}
for n in ['Decision tree','Random forest']:
    m=pipe(mk[n]).fit(Xtr,ytr); R['gap'][n+' (untuned)']=[float(roc_auc_score(ytr,m.predict_proba(Xtr)[:,1])),float(roc_auc_score(yte,m.predict_proba(Xte)[:,1]))]
print(R['gap'])
p=final.predict_proba(Xte)[:,1]
fig,axs=plt.subplots(1,3,figsize=(14,4.3))
ConfusionMatrixDisplay.from_predictions(yte,(p>=.5).astype(int),display_labels=['No','Yes'],ax=axs[0],colorbar=False,cmap='Blues'); axs[0].set_title('Confusion matrix (threshold 0.50)'); axs[0].set_xlabel('Predicted readmission'); axs[0].set_ylabel('Actual readmission'); axs[0].grid(False)
RocCurveDisplay.from_predictions(yte,p,ax=axs[1],name=best); axs[1].plot([0,1],[0,1],'k--',label='Random guess'); axs[1].legend(loc='lower right'); axs[1].set_title('ROC curve (test set)'); axs[1].set_xlabel('False positive rate'); axs[1].set_ylabel('True positive rate (recall)')
PrecisionRecallDisplay.from_predictions(yte,p,ax=axs[2],name=best); axs[2].axhline(yte.mean(),c='k',ls='--',label='Readmission rate (baseline)'); axs[2].legend(); axs[2].set_title('Precision-recall curve (test set)'); axs[2].set_xlabel('Recall'); axs[2].set_ylabel('Precision')
plt.tight_layout(); plt.savefig('figs/f10.png',dpi=150); plt.show()
pr,rc,th=precision_recall_curve(yte,p)
fig,ax=plt.subplots(figsize=(7,4.3)); ax.plot(th,pr[:-1],label='Precision'); ax.plot(th,rc[:-1],label='Recall'); ax.axvline(thr,c='grey',ls='--',label=f'Chosen threshold ({thr:.2f})'); ax.axhline(yte.mean(),c='k',ls=':',label='Readmission rate (28.8%)')
ax.set_xlabel('Decision threshold (predicted probability)'); ax.set_ylabel('Score'); ax.set_title('Precision and recall against decision threshold (test set)'); ax.legend(); plt.tight_layout(); plt.savefig('figs/f11.png',dpi=150); plt.show()
ts,trs,tes=learning_curve(clone(tuned[best]),Xtr,ytr,cv=cv,scoring='roc_auc',train_sizes=np.linspace(.1,1,8),random_state=42)
R['lc']={'sizes':ts.tolist(),'train':trs.mean(1).tolist(),'val':tes.mean(1).tolist()}
fig,ax=plt.subplots(figsize=(7,4.3)); ax.plot(ts,trs.mean(1),'o-',label='Training AUC'); ax.plot(ts,tes.mean(1),'o-',label='Cross-validation AUC'); ax.fill_between(ts,tes.mean(1)-tes.std(1),tes.mean(1)+tes.std(1),alpha=.2); ax.axhline(.5,c='k',ls='--',lw=.8,label='Random guess (0.50)')
ax.set_xlabel('Number of training admissions'); ax.set_ylabel('ROC-AUC'); ax.set_title(f'Learning curve ({best}, tuned)'); ax.set_ylim(.4,1.0); ax.legend(); plt.tight_layout(); plt.savefig('figs/f12.png',dpi=150); plt.show()
json.dump(R,open('results.json','w'),indent=1,default=str)

,acc,prec,rec,f1,auc
Baseline (always predict No),0.711667,0.0,0.0,0.0,0.5
Logistic regression (tuned),0.528333,0.296296,0.462428,0.361174,0.496149
Decision tree (tuned),0.496667,0.305136,0.583815,0.400794,0.515534
Random forest (tuned),0.586667,0.275449,0.265896,0.270588,0.477847


Final model at chosen threshold: {'acc': 0.4766666666666667, 'prec': 0.3025210084033613, 'rec': 0.6242774566473989, 'f1': 0.4075471698113208, 'auc': 0.5155338360114252, 'tn_fp_fn_tp': [178, 249, 65, 108]}


{'Logistic regression': [0.5307531757443884, 0.49614869163812597], 'Decision tree': [0.6043445586503583, 0.5155338360114252], 'Random forest': [0.8610694779411827, 0.4778465162242288], 'Decision tree (untuned)': [1.0, 0.5094489041707841], 'Random forest (untuned)': [1.0, 0.4841954217487241]}
